In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

C:\Users\1234\AppData\Local\Temp\ipykernel_19404\4159541439.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader


In [21]:
from pathlib import Path

In [2]:
load_dotenv(override=True)

True

In [10]:
openai_api_key = os.getenv("OPENAI_API_KEY")
MODEL = "gpt-4.1-nano"
db_name = "vector_db"

In [4]:
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY environment variable is not set. Please set it in your .env file.")
else:
    print(f"OPENAI_API_KEY is set, starts with {openai_api_key[:9]}")

OPENAI_API_KEY is set, starts with sk-proj-z


In [9]:
knowledgePath = "knowledge-base/**/*.md"
files = glob.glob(knowledgePath, recursive=True)
print(f"Found {len(files)} files in the knowledge base.")

full_knowledge = ""
for file_path in files:
    with open(file_path, "r", encoding="utf-8") as f:
        full_knowledge += f.read()
        full_knowledge += "\n\n"  # Add a separator between files

print(f"Total length of knowledge base: {len(full_knowledge):,} characters.")

Found 76 files in the knowledge base.
Total length of knowledge base: 304,434 characters.


In [11]:
encoding = tiktoken.encoding_for_model(MODEL)
tokens= encoding.encode(full_knowledge)
token_count = len(tokens)
print(f"total tokens for {MODEL} are {token_count:,} tokens.")


total tokens for gpt-4.1-nano are 63,555 tokens.


In [25]:
folders = glob.glob("knowledge-base/*")
documents = []
for folder in folders:
    doc_type= os.path.basename(folder)
    # print(doc_type)
    loader = DirectoryLoader(folder,glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
    folder_docs = loader.load()
    # print(folder_docs)
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)
# print(documents)
print(f"Loaded {len(documents)} documents from the knowledge base.")

Loaded 76 documents from the knowledge base.


In [26]:
documents[0]

Document(metadata={'source': 'knowledge-base\\company\\about.md', 'doc_type': 'company'}, page_content="# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a

In [27]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks.")


Split into 413 chunks.


In [28]:
chunks[9]

Document(metadata={'source': 'knowledge-base\\company\\careers.md', 'doc_type': 'company'}, page_content='### Operations & Support\n\n**Technical Support Specialist** - Remote\n- Provide tier 2/3 technical support to clients\n- Troubleshoot platform issues\n- Create documentation and knowledge base articles\n- 2+ years technical support experience\n\n**HR Business Partner** - San Francisco, CA\n- Partner with leadership on people strategy\n- Support talent development and retention\n- Drive culture and engagement initiatives\n- 4+ years HR experience, tech industry preferred\n\n## How to Apply\n\nVisit our careers portal at careers.insurellm.com or send your resume to jobs@insurellm.com. Please include the position title in your subject line.\n\nInsurellm is an equal opportunity employer. We celebrate diversity and are committed to creating an inclusive environment for all employees.')